# Re-DocRED (revised) — Baseline + DREAM 학습 / 평가

baseline(F1 최고)에 **DREAM(evidence-guided) 기법**만 얹은 `re_model_dream.DocREModelDREAM`을
`train_revised`로 학습, `dev_revised`로 평가. **best-checkpoint tracking + early-stopping(patience 5)** 적용.

**실행 전**: `data/docred_io.py`(revised split), `Scripts/atlop/re_model_dream.py`,
`Scripts/atlop/train_dream_es.py` 를 **dh에 push** 하세요.
(A100 GPU 런타임 권장. 위→아래 순서대로 실행)


## 1) Google Drive 마운트

In [22]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2) 코드(dh) + revised 데이터(main, LFS) 가져오기

In [23]:
%cd /content
!apt-get -qq install -y git-lfs > /dev/null 2>&1
!git lfs install --skip-repo
!rm -rf project1                          # 이전 clone 있으면 제거 (안 그러면 clone 스킵→옛 코드)
!git clone -q https://github.com/multiful/Information_Extraction.git project1
%cd /content/project1
!git checkout -q dh                       # DREAM 모델/스크립트 + docred_io(revised split)
# revised 데이터는 main에 LFS로 있음 -> 가져와 실체화
!git checkout origin/main -- docred_data/data/train_revised.json docred_data/data/dev_revised.json
!git lfs pull --include="docred_data/data/train_revised.json,docred_data/data/dev_revised.json"
!pip install -q transformers==4.57.6 accelerate

# docred_io에 revised split 보장 (스테일 clone/미push 대비 런타임 패치)
import re, pathlib
_p = pathlib.Path("data/docred_io.py"); _s = _p.read_text(encoding="utf-8")
if "train_revised" not in _s:
    _p.write_text(re.sub(r'SPLITS\s*=\s*\[[^\]]*\]',
        'SPLITS = ["train_annotated","train_distant","dev","test","train_revised","dev_revised","test_revised"]',
        _s, count=1), encoding="utf-8")
    print("docred_io SPLITS 런타임 패치")

import os
for fp in ["Scripts/atlop/re_model_dream.py", "Scripts/atlop/train_dream_es.py"]:
    assert os.path.exists(fp), f"{fp} 없음 -> dh에 push 했는지 확인!"
for f in ["train_revised", "dev_revised"]:
    kb = os.path.getsize(f"docred_data/data/{f}.json") // 1024
    print(f"{f}: {kb} KB", "OK" if kb > 100 else "  <-- LFS 실체화 실패")

/content
Git LFS initialized.
/content/project1
train_revised: 18166 KB OK
dev_revised: 3169 KB OK


## 3) 인코더(bert-base-cased) 로컬 다운로드 (stall 우회)

In [25]:
import os
os.makedirs("/content/bert-base-cased", exist_ok=True)
%cd /content/bert-base-cased
B  = "https://huggingface.co/google-bert/bert-base-cased/resolve/main"
UA = "Mozilla/5.0"
for f in ["config.json", "vocab.txt", "tokenizer.json", "tokenizer_config.json", "model.safetensors"]:
    print("↓", f, flush=True)
    !wget -c --tries=200 --timeout=30 --waitretry=3 --header="User-Agent: {UA}" -O {f} {B}/{f}
%cd /content/project1
!ls -lh /content/bert-base-cased/model.safetensors   # ~436M 이면 성공
# 계속 403/멈춤이면 B를 미러로: "https://hf-mirror.com/google-bert/bert-base-cased/resolve/main"

/content/bert-base-cased
↓ config.json
--2026-07-15 03:18:34--  https://huggingface.co/google-bert/bert-base-cased/resolve/main/config.json
Resolving huggingface.co (huggingface.co)... 108.157.142.74, 108.157.142.55, 108.157.142.53, ...
Connecting to huggingface.co (huggingface.co)|108.157.142.74|:443... connected.
HTTP request sent, awaiting response... 416 Range Not Satisfiable

    The file is already fully retrieved; nothing to do.

↓ vocab.txt
--2026-07-15 03:18:34--  https://huggingface.co/google-bert/bert-base-cased/resolve/main/vocab.txt
Resolving huggingface.co (huggingface.co)... 108.157.142.74, 108.157.142.55, 108.157.142.53, ...
Connecting to huggingface.co (huggingface.co)|108.157.142.74|:443... connected.
HTTP request sent, awaiting response... 416 Range Not Satisfiable

    The file is already fully retrieved; nothing to do.

↓ tokenizer.json
--2026-07-15 03:18:34--  https://huggingface.co/google-bert/bert-base-cased/resolve/main/tokenizer.json
Resolving huggingface.co (

## 4) DREAM 학습(train_revised) + 평가(dev_revised) — best-ckpt + early-stopping
- baseline + DREAM(evidence-guided), 분류기는 baseline bilinear 그대로
- `--patience 5` : dev_F1 5 epoch 개선 없으면 중단 / `--epochs 30` : 상한(ES가 조기종료)
- best epoch만 저장. 매 epoch `dev_F1 / Ign_F1` 출력, best 갱신 시 `*best*`
- (선택) baseline warm-start: `--init_ckpt results/atlop.pt` 추가 (atlop.pt를 Colab에 둔 경우)


In [26]:
!python -u -m Scripts.atlop.train_dream_es \
  --model_name_or_path /content/bert-base-cased \
  --train_split train_revised --dev_split dev_revised --distant_mode none \
  --epochs 30 --patience 5 --eval_batch_size 32 \
  --run_name atlop_dream_revised --save_model

[device] cuda  [model] dream (baseline + evidence-guided; +best-ckpt +early-stop patience=5, evi_lambda=0.1)
[data] train=3053 (train_revised) dev=500 (dev_revised) docs
preprocess-full: 100% 500/500 [00:04<00:00, 117.77it/s]
preprocess-full: 100% 3053/3053 [00:22<00:00, 137.18it/s]
2026-07-15 03:19:11.867356: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-15 03:19:11.939035: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
[main] annotated-train on 3053 docs (최대 30 epoch, patience 5)
  [annotated-train] epoch 0 

## 5) 결과 Drive 백업 (필수) — `dst`를 본인 경로로

In [27]:
dst = "/content/drive/MyDrive/MS_AI_NLP(2026)_실습자료/21_실전프로젝트1"   # <- 본인 경로
!cp results/atlop_dream_revised.pt "{dst}/"
!cp results/atlop_dream_revised_dev_predictions.json "{dst}/"
print("✅ Drive 저장 완료:", dst)

✅ Drive 저장 완료: /content/drive/MyDrive/MS_AI_NLP(2026)_실습자료/21_실전프로젝트1
